In [1]:
ADAPTER_PATH = "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
REFERENCE_NOTEBOOK_PATH = "/kaggle/input/notebooks/huikang/nvidia-nemotron-all-linear"
TRAIN_DATA_PATH = "/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"
TEST_GENERATION = True

RUN_NAME = "huikang-baseline-v1"
ENABLE_THINKING = True
SAVE_BASELINE_ARTIFACTS = True
BASELINE_ERROR_ROWS = 20
RUN_MAX_TOKENS_ABLATION = True
MAX_TOKENS_ABLATION = 7680
FINAL_ANSWER_INSTRUCTION = (
    "Reason step-by-step, but keep the reasoning concise and efficient.\n"
    "Avoid unnecessary repetition, excessive verification, or multiple alternative approaches.\n"
    "Once you determine the final answer, stop reasoning immediately.\n\n"

    "After completing your reasoning, output exactly one final answer "
    "inside a single LaTeX \\boxed{} command.\n\n"

    "Rules for the final boxed answer:\n"
    "- Use exactly one \\boxed{}\n"
    "- The box must contain only the final answer\n"
    "- Do not include variable names like x=\n"
    "- Do not include units\n"
    "- Do not include explanations inside the box\n"
    "- Do not include punctuation inside the box\n"
    "- Do not output multiple boxed answers\n"
    "- The final \\boxed{} must appear at the end of the response\n"
    "- Do not output any text after the final \\boxed{}"
)

In [2]:
import shutil

shutil.copytree(
    ADAPTER_PATH,
    "/kaggle/working/",
    dirs_exist_ok=True,
)

shutil.copytree(
    REFERENCE_NOTEBOOK_PATH,
    "/kaggle/working/reference",
    dirs_exist_ok=True,
)


'/kaggle/working/reference'

In [3]:
import zipfile

with zipfile.ZipFile("reference/submission.zip", "r") as zip_ref:
    zip_ref.extractall("reference")

# Compare configs

In [4]:
import json

with open("reference/adapter_config.json") as f:
    reference_adapter_config = json.load(f)

print(reference_adapter_config)

{'alora_invocation_tokens': None, 'alpha_pattern': {}, 'arrow_config': None, 'auto_mapping': None, 'base_model_name_or_path': '/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1', 'bias': 'none', 'corda_config': None, 'ensure_weight_tying': False, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': True, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 16, 'lora_bias': False, 'lora_dropout': 0.05, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'peft_version': '0.18.1', 'qalora_group_size': 16, 'r': 32, 'rank_pattern': {}, 'revision': None, 'target_modules': ['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj'], 'target_parameters': None, 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_qalora': False, 'use_r

In [5]:
import json

with open("adapter_config.json") as f:
    trained_adapter_config = json.load(f)

print(trained_adapter_config)

{'alpha_pattern': {}, 'auto_mapping': None, 'base_model_name_or_path': None, 'bias': 'none', 'corda_config': None, 'eva_config': None, 'exclude_modules': None, 'fan_in_fan_out': False, 'inference_mode': False, 'init_lora_weights': True, 'layer_replication': None, 'layers_pattern': None, 'layers_to_transform': None, 'loftq_config': {}, 'lora_alpha': 32, 'lora_bias': False, 'lora_dropout': 0, 'megatron_config': None, 'megatron_core': 'megatron.core', 'modules_to_save': None, 'peft_type': 'LORA', 'r': 32, 'rank_pattern': {}, 'revision': None, 'target_modules': 'all-linear', 'task_type': 'CAUSAL_LM', 'trainable_token_indices': None, 'use_dora': False, 'use_rslora': False}


In [6]:
for k, reference_value in reference_adapter_config.items():
    if k in trained_adapter_config and reference_value != trained_adapter_config[k]:
        print(k)
        print(reference_value)
        print(trained_adapter_config[k])
        print()

base_model_name_or_path
/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1
None

inference_mode
True
False

lora_alpha
16
32

lora_dropout
0.05
0

target_modules
['k_proj', 'o_proj', 'in_proj', 'q_proj', 'up_proj', 'v_proj', 'down_proj', 'out_proj']
all-linear



In [7]:
for k, reference_value in reference_adapter_config.items():
    if k not in trained_adapter_config:
        print(k)
        print(reference_value)
        print()

alora_invocation_tokens
None

arrow_config
None

ensure_weight_tying
False

peft_version
0.18.1

qalora_group_size
16

target_parameters
None

use_qalora
False



In [8]:
for k, v in trained_adapter_config.items():
    if k not in reference_adapter_config:
        print(k)

# Align configs

In [9]:
trained_adapter_config["target_modules"] = [
    "k_proj",
    "o_proj",
    "in_proj",
    "q_proj",
    "up_proj",
    "v_proj",
    "down_proj",
    "out_proj",
    "lm_head",
]

with open("adapter_config.json", "w") as f:
    f.write(json.dumps(trained_adapter_config))

In [10]:
with open("adapter_config.json") as f:
    print(f.read())

{"alpha_pattern": {}, "auto_mapping": null, "base_model_name_or_path": null, "bias": "none", "corda_config": null, "eva_config": null, "exclude_modules": null, "fan_in_fan_out": false, "inference_mode": false, "init_lora_weights": true, "layer_replication": null, "layers_pattern": null, "layers_to_transform": null, "loftq_config": {}, "lora_alpha": 32, "lora_bias": false, "lora_dropout": 0, "megatron_config": null, "megatron_core": "megatron.core", "modules_to_save": null, "peft_type": "LORA", "r": 32, "rank_pattern": {}, "revision": null, "target_modules": ["k_proj", "o_proj", "in_proj", "q_proj", "up_proj", "v_proj", "down_proj", "out_proj", "lm_head"], "task_type": "CAUSAL_LM", "trainable_token_indices": null, "use_dora": false, "use_rslora": false}


# Compare adapters

In [11]:
def trained_adapter_key_rename(key_name: str) -> str:
    key_name = key_name.replace("base_model.model.model", "base_model.model.backbone")
    return key_name

In [12]:
from safetensors import safe_open

trained_adapter_keys = set()
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        trained_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [13]:
from safetensors import safe_open

reference_adapter_keys = set()
with safe_open(
    "reference/adapter_model.safetensors", framework="pt", device="cpu"
) as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        reference_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [14]:
from safetensors import safe_open
import glob

model_keys = set()
for model_safetensors in glob.glob(
    "/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1/*.safetensors"
):
    with safe_open(model_safetensors, framework="pt", device="cpu") as f:
        for key in f.keys():
            tensor_slice = f.get_slice(key)
            model_keys.add(
                (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
            )

In [15]:
# set([".".join(x.split(".")[3:]) for x in trained_adapter_keys]) - set([".".join(x.split(".")[3:]) for x in reference_adapter_keys])

In [16]:
sorted(model_keys)[-20:]

[('backbone.layers.8.mixer.experts.98.down_proj.weight', (2688, 1856), 'BF16'),
 ('backbone.layers.8.mixer.experts.98.up_proj.weight', (1856, 2688), 'BF16'),
 ('backbone.layers.8.mixer.experts.99.down_proj.weight', (2688, 1856), 'BF16'),
 ('backbone.layers.8.mixer.experts.99.up_proj.weight', (1856, 2688), 'BF16'),
 ('backbone.layers.8.mixer.gate.e_score_correction_bias', (128,), 'F32'),
 ('backbone.layers.8.mixer.gate.weight', (128, 2688), 'BF16'),
 ('backbone.layers.8.mixer.shared_experts.down_proj.weight',
  (2688, 3712),
  'BF16'),
 ('backbone.layers.8.mixer.shared_experts.up_proj.weight',
  (3712, 2688),
  'BF16'),
 ('backbone.layers.8.norm.weight', (2688,), 'BF16'),
 ('backbone.layers.9.mixer.A_log', (64,), 'F32'),
 ('backbone.layers.9.mixer.D', (64,), 'F32'),
 ('backbone.layers.9.mixer.conv1d.bias', (6144,), 'BF16'),
 ('backbone.layers.9.mixer.conv1d.weight', (6144, 1, 4), 'BF16'),
 ('backbone.layers.9.mixer.dt_bias', (64,), 'BF16'),
 ('backbone.layers.9.mixer.in_proj.weight', (1

In [17]:
sorted(reference_adapter_keys)[-20:]

[('base_model.model.backbone.layers.8.mixer.experts.97.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_pro

In [18]:
sorted(trained_adapter_keys)[-20:]

[('base_model.model.model.layers.7.mixer.x_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.model.layers.7.mixer.x_proj.lora_B.weight',
  (4096, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w1.lora_A.weight',
  (1, 32, 2688),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w1.lora_B.weight',
  (128, 1856, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w2.lora_A.weight',
  (128, 32, 1856),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w2.lora_B.weight',
  (1, 2688, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w3.lora_A.weight',
  (0,),
  'F32'),
 ('base_model.model.model.layers.8.mixer.experts.w3.lora_B.weight',
  (0,),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.down_proj.lora_A.weight',
  (32, 3712),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.model.layers.8.mixer.shared_experts.

In [19]:
len(trained_adapter_keys), len(reference_adapter_keys)

(418, 12008)

In [20]:
(
    len(trained_adapter_keys - reference_adapter_keys),
    len(reference_adapter_keys - trained_adapter_keys),
)

(418, 12008)

# Update adapter

In [21]:
import re

import torch
from safetensors import safe_open
from safetensors.torch import save_file

# --- Load all trained adapter tensors ---
adapter_tensors = {}
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        adapter_tensors[key] = f.get_tensor(key)

# --- Collect adapter base names (without .lora_A/.lora_B.weight suffix) ---
base_names = set()
for key in adapter_tensors:
    base = re.sub(r"\.lora_[AB]\.weight$", "", key)
    base_names.add(base)

# --- Identify Mamba layers needing gate_proj+x_proj → in_proj ---
mamba_merge_layers = {}  # layer_path -> {"gate_proj": base, "x_proj": base}
for base in base_names:
    for proj in ("gate_proj", "x_proj"):
        if f".{proj}" in base:
            layer_path = base.rsplit(f".{proj}", 1)[0]
            mamba_merge_layers.setdefault(layer_path, {})[proj] = base
mamba_merge_bases = set()
for projs in mamba_merge_layers.values():
    mamba_merge_bases.update(projs.values())

# --- Build model_key_shapes for in_proj dimension lookup ---
model_key_shapes = {k: s for k, s, _ in model_keys}

# --- Build output tensors ---
tensors = {}

for base in sorted(base_names):
    lora_A = adapter_tensors[f"{base}.lora_A.weight"]
    lora_B = adapter_tensors[f"{base}.lora_B.weight"]
    renamed = trained_adapter_key_rename(base)

    # Skip empty w3 experts
    if ".experts.w3" in base and lora_A.numel() == 0:
        continue

    # # Skip lm_head (not in reference adapter)
    # if ".lm_head" in base:
    #     continue

    # Skip gate_proj/x_proj — handled in Mamba merge pass below
    if base in mamba_merge_bases:
        continue

    # --- Expert unfusing: w1 → per-expert up_proj, w2 → per-expert down_proj ---
    if ".experts.w1" in base or ".experts.w2" in base:
        # Broadcast shared dimension (one of A/B has shape[0]==1)
        # Use expand + contiguous to avoid shared memory in safetensors
        if lora_A.shape[0] == 1:
            lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
        elif lora_B.shape[0] == 1:
            lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()

        num_experts = lora_A.shape[0]
        proj_name = "up_proj" if ".w1" in base else "down_proj"

        for i in range(num_experts):
            exp_renamed = re.sub(
                r"\.experts\.w[12]",
                f".experts.{i}.{proj_name}",
                renamed,
            )
            tensors[f"{exp_renamed}.lora_A.weight"] = lora_A[i].contiguous()
            tensors[f"{exp_renamed}.lora_B.weight"] = lora_B[i].contiguous()
        continue

    # --- Direct rename for everything else ---
    tensors[f"{renamed}.lora_A.weight"] = lora_A
    tensors[f"{renamed}.lora_B.weight"] = lora_B

# --- Mamba: gate_proj + x_proj → in_proj via SVD ---
for layer_path, projs in sorted(mamba_merge_layers.items()):
    renamed_layer = trained_adapter_key_rename(layer_path)
    in_proj_base = f"{renamed_layer}.in_proj"

    model_in_proj_key = (
        renamed_layer.replace("base_model.model.", "") + ".in_proj.weight"
    )
    in_proj_dim = model_key_shapes[model_in_proj_key][0]

    gate_A = adapter_tensors[f"{projs['gate_proj']}.lora_A.weight"].float()
    gate_B = adapter_tensors[f"{projs['gate_proj']}.lora_B.weight"].float()
    x_A = adapter_tensors[f"{projs['x_proj']}.lora_A.weight"].float()
    x_B = adapter_tensors[f"{projs['x_proj']}.lora_B.weight"].float()
    rank = gate_A.shape[0]

    # Build combined rank-64 representation, then SVD to best rank-32
    A_cat = torch.cat([gate_A, x_A], dim=0)  # (64, in_dim)
    B_block = torch.zeros(in_proj_dim, 2 * rank)
    B_block[: gate_B.shape[0], :rank] = gate_B
    B_block[gate_B.shape[0] : gate_B.shape[0] + x_B.shape[0], rank:] = x_B

    Q_B, R_B = torch.linalg.qr(B_block)
    Q_A, R_A = torch.linalg.qr(A_cat.T)
    core = R_B @ R_A.T
    U, S, Vh = torch.linalg.svd(core, full_matrices=False)

    k = rank
    new_B = (Q_B @ U[:, :k]) * S[:k].unsqueeze(0)
    new_A = Vh[:k, :] @ Q_A.T

    kept = S[:k].sum().item()
    total = S.sum().item()
    print(
        f"{layer_path}: SVD kept {kept:.2f}/{total:.2f} "
        f"({kept / total * 100:.1f}%) of singular value mass"
    )

    tensors[f"{in_proj_base}.lora_A.weight"] = new_A
    tensors[f"{in_proj_base}.lora_B.weight"] = new_B

print(
    f"\nConverted {len(adapter_tensors)} trained tensors → {len(tensors)} output tensors"
)
save_file(tensors, "adapter_model.safetensors")

base_model.model.model.layers.0.mixer: SVD kept 4.48/5.94 (75.4%) of singular value mass
base_model.model.model.layers.11.mixer: SVD kept 3.93/5.23 (75.2%) of singular value mass
base_model.model.model.layers.14.mixer: SVD kept 3.98/5.31 (75.0%) of singular value mass
base_model.model.model.layers.16.mixer: SVD kept 4.06/5.39 (75.3%) of singular value mass
base_model.model.model.layers.18.mixer: SVD kept 3.95/5.25 (75.2%) of singular value mass
base_model.model.model.layers.2.mixer: SVD kept 3.98/5.09 (78.1%) of singular value mass
base_model.model.model.layers.21.mixer: SVD kept 4.04/5.35 (75.4%) of singular value mass
base_model.model.model.layers.23.mixer: SVD kept 4.15/5.50 (75.4%) of singular value mass
base_model.model.model.layers.25.mixer: SVD kept 4.20/5.66 (74.3%) of singular value mass
base_model.model.model.layers.28.mixer: SVD kept 4.58/5.97 (76.8%) of singular value mass
base_model.model.model.layers.30.mixer: SVD kept 4.78/6.26 (76.5%) of singular value mass
base_model.m

In [22]:
from safetensors import safe_open

updated_adapter_keys = set()
with safe_open("adapter_model.safetensors", framework="pt", device="cpu") as f:
    for key in f.keys():
        tensor_slice = f.get_slice(key)
        updated_adapter_keys.add(
            (key, tuple(tensor_slice.get_shape()), tensor_slice.get_dtype())
        )

In [23]:
sorted(updated_adapter_keys)[-20:]

[('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.97.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.98.up_proj.lora_B.weight',
  (1856, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_A.weight',
  (32, 1856),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.down_proj.lora_B.weight',
  (2688, 32),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.up_proj.lora_A.weight',
  (32, 2688),
  'F32'),
 ('base_model.model.backbone.layers.8.mixer.experts.99.up_proj.lo

In [24]:
len(updated_adapter_keys), len(reference_adapter_keys)

(12010, 12008)

In [25]:
(
    len(updated_adapter_keys - reference_adapter_keys),
    len(reference_adapter_keys - updated_adapter_keys),
)

(2, 0)

In [26]:
sorted(reference_adapter_keys - updated_adapter_keys)[:20]

[]

In [27]:
sorted(updated_adapter_keys - reference_adapter_keys)[:20]

[('base_model.model.backbone.lm_head.lora_A.weight', (32, 2688), 'F32'),
 ('base_model.model.backbone.lm_head.lora_B.weight', (131072, 32), 'F32')]

# Load model

In [28]:
"""Metric for NVIDIA (129716)."""

import subprocess
import sys

# Set up environment
commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]
if TEST_GENERATION:
    for cmd in commands:
        print(f"Running: {cmd}")
        subprocess.run(cmd, shell=True, check=True)
sys.path.insert(0, "/tmp")

Running: uv pip uninstall torch torchvision torchaudio


Using Python 3.12.12 environment at: /usr
Uninstalled 3 packages in 5.75s
 - torch==2.9.0+cpu
 - torchaudio==2.9.0+cpu
 - torchvision==0.24.0+cpu


Running: tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas
Running: chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell


In [29]:
import glob
import json
import math
import multiprocessing
import os
import re
import time
from pathlib import Path

import kagglehub
import pandas as pd

# Cloud resource configuration
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
DATA_PATH_CANDIDATES = [
    Path("/kaggle/input/competition_evaluation"),
    Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge"),
]
DATA_PATH = next(
    (candidate for candidate in DATA_PATH_CANDIDATES if candidate.exists()),
    DATA_PATH_CANDIDATES[-1],
)


In [30]:
class ParticipantVisibleError(Exception):
    pass


def cache_model(
    path: str | Path,
    exts: tuple[str, ...] = (".bin", ".pt", ".safetensors"),
    num_workers: int | None = None,
    chunk_mb: int = 256,
) -> int:
    """Pre-read model weight files into the OS page cache to speed up later loads."""
    from concurrent.futures import ThreadPoolExecutor, as_completed

    def warmup_file(fpath: Path) -> tuple[Path, int]:
        chunk_size = chunk_mb * 1024 * 1024
        total = 0
        try:
            with open(fpath, "rb") as f:
                while True:
                    data = f.read(chunk_size)
                    if not data:
                        break
                    total += len(data)
        except Exception as exc:
            print(f"Error reading {fpath}: {exc}")
        return fpath, total

    path = Path(path)
    if path.is_dir():
        files = [p for p in path.rglob("*") if p.is_file() and str(p).endswith(exts)]
        files.sort()
    else:
        files = [path] if path.exists() else []

    if not files:
        print(f"No model files found to cache at: {path}")
        return 0

    if num_workers is None:
        try:
            num_workers = min(multiprocessing.cpu_count(), 8)
        except Exception:
            num_workers = 4

    print(f"[cache_model] {len(files)} file(s), {num_workers} worker(s)")
    t0 = time.time()
    total_bytes = 0
    with ThreadPoolExecutor(max_workers=num_workers) as pool:
        futures = {pool.submit(warmup_file, f): f for f in files}
        for i, fut in enumerate(as_completed(futures), 1):
            fpath, n = fut.result()
            total_bytes += n
            print(f"[{i}/{len(files)}] cached {fpath.name}")

    elapsed = time.time() - t0
    gb = total_bytes / 1024**3
    speed = gb / elapsed if elapsed > 0 else 0
    print(f"[cache_model] total read {gb:.2f} GB")
    print(f"[cache_model] elapsed {elapsed:.2f} s, ~{speed:.2f} GB/s")
    return total_bytes


def clean_extracted_answer(text: str | None) -> str:
    if text is None:
        return "NOT_FOUND"

    cleaned = str(text).strip()
    cleaned = cleaned.strip("`").strip()
    cleaned = re.sub(r"^\$+|\$+$", "", cleaned)
    cleaned = re.sub(r"^\\text\{([^}]*)\}$", r"\1", cleaned)
    cleaned = re.sub(r"^\\mathrm\{([^}]*)\}$", r"\1", cleaned)
    cleaned = re.sub(r"^\\boxed\{([^}]*)\}$", r"\1", cleaned)

    if cleaned.startswith("{") and cleaned.endswith("}") and len(cleaned) > 1:
        cleaned = cleaned[1:-1].strip()

    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    cleaned = cleaned.rstrip(".,;: ")
    return cleaned or "NOT_FOUND"


def extract_final_answer(text: str | None) -> str:
    r"""Extract the final answer from a model response."""
    if text is None:
        return "NOT_FOUND"

    text = text.replace("\r\n", "\n")

    boxed_matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    if boxed_matches:
        return clean_extracted_answer(boxed_matches[-1])

    patterns = [
        r"The final answer is\s*[:=-]\s*([^\n]+)",
        r"Final answer is\s*[:=-]\s*([^\n]+)",
        r"Final answer\s*[:=-]\s*([^\n]+)",
        r"Answer\s*[:=-]\s*([^\n]+)",
        r"The answer is\s*[:=-]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return clean_extracted_answer(matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        cleaned_line = clean_extracted_answer(line)
        if cleaned_line == "NOT_FOUND":
            continue
        if len(cleaned_line) <= 64:
            return cleaned_line

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return clean_extracted_answer(number_matches[-1])

    return "NOT_FOUND"


def verify(stored_answer: str, predicted: str) -> bool:
    """Verify whether a prediction matches the stored answer."""
    stored_answer = clean_extracted_answer(stored_answer)
    predicted = clean_extracted_answer(predicted)

    try:
        stored_num = float(stored_answer)
        predicted_num = float(predicted)
        return math.isclose(stored_num, predicted_num, rel_tol=1e-2, abs_tol=1e-5)
    except Exception:
        return predicted.lower() == stored_answer.lower()


def build_user_content(problem_text: str) -> str:
    return problem_text.rstrip() + "\n" + FINAL_ANSWER_INSTRUCTION


def build_prompt(tokenizer, problem_text: str, enable_thinking: bool = True) -> str:
    user_content = build_user_content(problem_text)
    try:
        return tokenizer.apply_chat_template(
            [{"role": "user", "content": user_content}],
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=enable_thinking,
        )
    except Exception:
        return user_content


def build_prompts(
    problem_texts: list[str],
    tokenizer,
    enable_thinking: bool = True,
) -> list[str]:
    return [
        build_prompt(tokenizer, problem_text, enable_thinking=enable_thinking)
        for problem_text in problem_texts
    ]


def find_lora_path(extra_dirs: list[str] | None = None) -> str:
    search_dirs = ["/kaggle/tmp", "/kaggle/working"]
    if extra_dirs:
        search_dirs.extend(extra_dirs)

    adapter_configs = []
    for search_dir in search_dirs:
        if not os.path.exists(search_dir):
            continue
        adapter_configs.extend(
            glob.glob(os.path.join(search_dir, "**/adapter_config.json"), recursive=True)
        )

    adapter_configs = sorted(
        {
            path
            for path in adapter_configs
            if "/reference/" not in path.replace("\\", "/")
        },
        key=lambda path: (
            path.replace("\\", "/") != "/kaggle/working/adapter_config.json",
            len(path),
            path,
        ),
    )
    if not adapter_configs:
        raise ParticipantVisibleError("No adapter_config.json found outside the reference directory.")

    return os.path.dirname(adapter_configs[0])


def summarize_eval_df(
    eval_df: pd.DataFrame,
    lora_path: str,
    experiment_config: dict,
) -> dict:
    correct_series = eval_df["correct"] if "correct" in eval_df.columns else pd.Series(dtype=bool)
    accuracy = float(correct_series.mean()) if not correct_series.empty else None
    error_count = int((~correct_series).sum()) if not correct_series.empty else None

    summary = {
        "run_name": experiment_config["run_name"],
        "rows": int(len(eval_df)),
        "accuracy": accuracy,
        "errors": error_count,
        "lora_path": lora_path,
        "config": experiment_config,
    }
    if "task" in eval_df.columns and not correct_series.empty:
        task_accuracy = eval_df.groupby("task")["correct"].mean().sort_values().to_dict()
        summary["task_accuracy"] = {
            task: float(score) for task, score in task_accuracy.items()
        }
    return summary


def save_local_eval_artifacts(
    eval_df: pd.DataFrame,
    lora_path: str,
    experiment_config: dict,
    artifact_dir: str | Path = "local_eval",
    error_rows: int = 20,
    write_legacy_files: bool = False,
) -> dict:
    artifact_dir = Path(artifact_dir)
    artifact_dir.mkdir(parents=True, exist_ok=True)

    run_name = experiment_config["run_name"]
    summary = summarize_eval_df(eval_df, lora_path=lora_path, experiment_config=experiment_config)

    predictions_path = artifact_dir / f"{run_name}_predictions.csv"
    summary_path = artifact_dir / f"{run_name}_summary.json"
    errors_path = artifact_dir / f"{run_name}_errors.csv"

    eval_df.to_csv(predictions_path, index=False)
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    error_columns = [
        column
        for column in ["id", "task", "answer", "predicted", "output", "prompt"]
        if column in eval_df.columns
    ]
    eval_df.loc[~eval_df["correct"], error_columns].head(error_rows).to_csv(
        errors_path,
        index=False,
    )

    if write_legacy_files:
        eval_df.to_csv("predictions.csv", index=False)
        Path("baseline_summary.json").write_text(
            json.dumps(summary, indent=2),
            encoding="utf-8",
        )
        eval_df.loc[~eval_df["correct"], error_columns].head(error_rows).to_csv(
            "baseline_errors.csv",
            index=False,
        )

    return summary


def run_local_experiments(
    source_df: pd.DataFrame,
    llm,
    lora_path: str,
    prompts: list[str],
    experiment_configs: list[dict],
    primary_run_name: str,
    error_rows: int = 20,
) -> tuple[dict[str, dict], pd.DataFrame]:
    run_results = {}
    comparison_rows = []

    for experiment_config in experiment_configs:
        sampling_params = SamplingParams(
            temperature=experiment_config["temperature"],
            top_p=experiment_config["top_p"],
            max_tokens=experiment_config["max_tokens"],
        )

        outputs = llm.generate(
            prompts,
            sampling_params=sampling_params,
            lora_request=LoRARequest("adapter", 1, lora_path),
        )

        eval_df = source_df.copy()
        
        from collections import Counter

        def majority_vote_from_output(output):
            answers = []
            for o in output.outputs:
                ans = extract_final_answer(o.text)
                if ans != "NOT_FOUND":
                    answers.append(ans)
            if not answers:
                return "NOT_FOUND"
            return Counter(answers).most_common(1)[0][0]
        
        eval_df["output"] = [output.outputs[0].text for output in outputs]
        eval_df["predicted"] = [majority_vote_from_output(output) for output in outputs]
        
        eval_df["correct"] = eval_df.apply(
            lambda row: verify(str(row["answer"]), str(row["predicted"])),
            axis=1,
        )
        eval_df["raw_output_chars"] = eval_df["output"].str.len()

        summary = save_local_eval_artifacts(
            eval_df,
            lora_path=lora_path,
            experiment_config=experiment_config,
            error_rows=error_rows,
            write_legacy_files=experiment_config["run_name"] == primary_run_name,
        )

        comparison_rows.append(
            {
                "run_name": experiment_config["run_name"],
                "max_tokens": experiment_config["max_tokens"],
                "temperature": experiment_config["temperature"],
                "top_p": experiment_config["top_p"],
                "enable_thinking": experiment_config["enable_thinking"],
                "accuracy": summary["accuracy"],
                "errors": summary["errors"],
                "rows": summary["rows"],
            }
        )
        run_results[experiment_config["run_name"]] = {
            "eval_df": eval_df,
            "summary": summary,
        }
        print(
            {
                "run_name": experiment_config["run_name"],
                "max_tokens": experiment_config["max_tokens"],
                "accuracy": summary["accuracy"],
                "errors": summary["errors"],
            }
        )

    comparison_df = pd.DataFrame(comparison_rows).sort_values(
        by=["accuracy", "max_tokens"],
        ascending=[False, True],
    )
    artifact_dir = Path("local_eval")
    artifact_dir.mkdir(parents=True, exist_ok=True)
    comparison_df.to_csv(artifact_dir / "experiment_compare.csv", index=False)
    return run_results, comparison_df


def generate_standard_submission(submission_dir: str):
    """Process an extracted submission archive to produce a standard submission file."""
    lora_path = find_lora_path(extra_dirs=[submission_dir])
    test_df = pd.read_csv(DATA_PATH / "test.csv", index_col=None)

    row_id_col = str(test_df.columns.to_list()[0])
    predictions = []
    for item in test_df.itertuples(index=False):
        predictions.append(
            {
                row_id_col: getattr(item, row_id_col),
                "prediction": lora_path,
            }
        )

    submission_df = pd.DataFrame(predictions)
    submission_df.to_csv("submission.csv", index=False)


def generate_predictions(
    test_df: pd.DataFrame,
    lora_path: str,
    row_id_col: str,
    max_lora_rank: int,
    max_tokens: int,
    top_p: float,
    temperature: float,
    max_num_seqs: int,
    gpu_memory_utilization: float,
    max_model_len: int,
    debug: bool = False,
) -> pd.DataFrame:
    """Load the model and generate predictions for the provided test data."""
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["TRANSFORMERS_NO_FLAX"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

    from vllm import LLM, SamplingParams
    from vllm.lora.request import LoRARequest

    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        max_tokens=max_tokens,
    )

    tokenizer = llm.get_tokenizer()
    prompts = build_prompts(
        list(test_df["prompt"]),
        tokenizer,
        enable_thinking=ENABLE_THINKING,
    )

    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        lora_request=LoRARequest("adapter", 1, lora_path),
    )

    predictions = []
    debug_records = []
    for item, output in zip(test_df.itertuples(index=False), outputs):
        raw_text = output.outputs[0].text
        extracted_answer = extract_final_answer(raw_text)
        row_id_val = getattr(item, row_id_col)

        predictions.append(
            {
                row_id_col: row_id_val,
                "prediction": extracted_answer,
            }
        )

        if debug:
            debug_records.append(
                {
                    row_id_col: row_id_val,
                    "raw_output": raw_text,
                    "extracted_prediction": extracted_answer,
                }
            )

    if debug and debug_records:
        pd.DataFrame(debug_records).to_csv("debug_predictions.csv", index=False)
        print("Debug data saved to debug_predictions.csv")

    return pd.DataFrame(predictions)


def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    max_lora_rank: int = 32,
    max_tokens: int = 7680,
    top_p: float = 1.0,
    temperature: float = 0.0,
    max_num_seqs: int = 64,
    gpu_memory_utilization: float = 0.85,
    max_model_len: int = 8192,
    debug: bool = False,
) -> float:
    r"""Evaluate the generated predictions against the ground truth."""
    lora_path = submission["prediction"].iloc[0]

    test_df = pd.read_csv(DATA_PATH / "test.csv", index_col=None)
    row_id_col = str(test_df.columns.to_list()[0])
    test_df = test_df[test_df[row_id_col].isin(solution[row_id_column_name])]

    submission = generate_predictions(
        test_df=test_df,
        lora_path=lora_path,
        row_id_col=row_id_column_name,
        max_lora_rank=max_lora_rank,
        max_tokens=max_tokens,
        top_p=top_p,
        temperature=temperature,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        max_model_len=max_model_len,
        debug=debug,
    )

    dataset = solution.merge(submission, on=row_id_column_name)
    num_correct = 0
    for item in dataset.itertuples(index=False):
        if verify(str(item.answer), str(item.prediction)):
            num_correct += 1

    return float(num_correct / len(solution))


In [31]:
# Cache Model
if TEST_GENERATION:
    cache_model(MODEL_PATH, num_workers=16, chunk_mb=1024)

[cache_model] 13 file(s), 16 worker(s)


# Init vLLM

In [ ]:
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

In [ ]:
# Keep first-round experiments single-variable and comparable.
max_model_len = 8192
max_lora_rank = 32
max_tokens = 7680
top_p = 0.95
temperature = 0.2
max_num_seqs = 64
gpu_memory_utilization = 0.85

BASE_EXPERIMENT_CONFIG = {
    "run_name": RUN_NAME,
    "adapter_path": ADAPTER_PATH,
    "max_model_len": max_model_len,
    "max_lora_rank": max_lora_rank,
    "max_tokens": max_tokens,
    "top_p": top_p,
    "temperature": temperature,
    "max_num_seqs": max_num_seqs,
    "gpu_memory_utilization": gpu_memory_utilization,
    "enable_thinking": ENABLE_THINKING,
}

LOCAL_EXPERIMENTS = [BASE_EXPERIMENT_CONFIG]
if RUN_MAX_TOKENS_ABLATION:
    LOCAL_EXPERIMENTS.append(
        {
            **BASE_EXPERIMENT_CONFIG,
            "run_name": f"{RUN_NAME}-max{MAX_TOKENS_ABLATION}",
            "max_tokens": MAX_TOKENS_ABLATION,
        }
    )

PRIMARY_RUN_NAME = BASE_EXPERIMENT_CONFIG["run_name"]
EXPERIMENT_CONFIG = BASE_EXPERIMENT_CONFIG

pd.DataFrame(
    [
        {
            "run_name": experiment["run_name"],
            "max_tokens": experiment["max_tokens"],
            "temperature": experiment["temperature"],
            "top_p": experiment["top_p"],
        }
        for experiment in LOCAL_EXPERIMENTS
    ]
)


In [ ]:
# Initialize vLLM Offline inference Engine

if TEST_GENERATION:
    llm = LLM(
        model=str(MODEL_PATH),
        tensor_parallel_size=1,
        max_num_seqs=max_num_seqs,
        gpu_memory_utilization=gpu_memory_utilization,
        dtype="auto",
        max_model_len=max_model_len,
        trust_remote_code=True,
        enable_lora=True,
        max_lora_rank=max_lora_rank,
        enable_prefix_caching=True,
        enable_chunked_prefill=True,
    )

    sampling_params = SamplingParams(
        temperature=temperature,
        top_p=top_p,
        n=3,
        max_tokens=max_tokens,
    )

# Test generation

In [ ]:
import pandas as pd

df = pd.read_csv(TRAIN_DATA_PATH)

problem_set = {
    # bit_manipulation
    "836b85e8", "b20b39bf", "af358750", "3f9bd1e7", "0528d502", "9992bbd0", "812131f1", "84e3f9f7",
    "114a41e3", "585f2ff6", "ea6859d7", "2fa48efe", "5f76ba09", "6a186446", "5d0db0d2", "75898981",
    "c7a37cda", "989dde0a", "d623e937", "100e280a", "f6a95641", "302dc36e", "3c9b8e0e", "b6e4a36d",
    "bd214050", "d50683b4", "f7346f0c", "5dd3345c", "dfc4839c", "0e7a6920", "31a4c9ef", "f8fc43d2",
    "19f4b3d6", "093de4ea", "9bd65991", "c6fa3e3f", "b8e0c853", "cb3317fe", "4ada9150", "c90fa3a6",
    # cipher
    "cf821623", "31c72d27", "73cd9008", "3975d230", "4db54201", "c1ffb3ac", "2a6c343e", "0e46fd1d",
    "49b244e3", "7b8e4432", "3019f44e", "4f8f23d6", "1fcbbb93", "987a223b", "84d10c70", "0dad87bf",
    # cryptarithm_deduce
    "1c7a0091", "ed61a9d6", "02b8d816", "d6c03e21", "2f6531cb", "a4ee9fa6", "02a04b59", "81b6d789",
    "bc83b0a1", "b1b10e83", "dea42835", "2c017f70", "b13d511a", "6897f05e", "64d775e5", "24e1f1d5",
    # cryptarithm_guess
    "c7844441", "0fcf912a", "07b440f0", "25ee72c3", "9dfe5ac9", "deed3497", "258b796b", "0da1841f",
    "e38c423b", "2e9973b7", "55f4fa64",
    # equation_numeric_deduce
    "91488dc9", "7e2e8a95", "35a89469", "a04ecffd", "27cec7a9", "d6a2e332", "04322d27", "8bc6a26c",
    "5c743e8a", "30763ac0", "91b42a45", "fecad63c", "c857a727", "b69391b8", "45df54db", "e5956ffa",
    # equation_numeric_guess
    "662fd21c", "e9e6b620", "8ae8e12a", "dc178d1c", "9c91b226", "c763054a", "66a0856f", "ef1b13ac",
    "8df3daad", "e7b87b82", "5d89a09c", "be877da5", "7c0c5227", "69fe4b0d", "b7ad0671", "4e840a1a",
    # gravity
    "d0dd2df7", "74a50b2c", "f9f20a7a", "ef3c7703", "85bc954c", "8de7d8bc", "22bb13b8", "87eb7ce0",
    "8a24aef9", "3c53c8af", "a6f1b553", "7b47f88d", "c0f6e1b8", "44dbe7d3", "827a6b1b", "6f3a0625",
    # numeral
    "3b4ebafd", "ad6ff612", "5a6ed2bf", "0adca57b", "d79d0cfd", "e6b04620", "f47276a4", "1e2de753",
    "0122d53a", "685bb0b1", "797ae611", "588a4ce8", "f19ffbf1", "8c281ee9", "972ef18a", "5092f0e0",
    # unit_conversion
    "082c1a06", "3dcaf042", "87342969", "8e1cff16", "d566ff0e", "598af975", "51a22965", "d3d82844",
    "e6157d05", "cd1280b0", "bbb61c3a", "740e0460", "be2416ec", "63ec749f", "26e6819a", "99948ad9",
}
df = df[df.id.isin(problem_set)].copy()

print(
    {
        "run_name": RUN_NAME,
        "local_eval_rows": int(len(df)),
        "columns": list(df.columns),
    }
)


In [ ]:
problem_texts = list(df["prompt"])

if TEST_GENERATION:
    tokenizer = llm.get_tokenizer()
    prompts = build_prompts(
        problem_texts,
        tokenizer,
        enable_thinking=ENABLE_THINKING,
    )


In [ ]:
# Generate predictions using continuous batching (without adapter, for baseline comparison)
if TEST_GENERATION:
    # outputs = llm.generate(
    #     prompts,
    #     sampling_params=sampling_params,
    # )
    pass

In [ ]:
print(os.listdir("."))

In [ ]:
lora_path = find_lora_path()

In [ ]:
print(os.listdir("/kaggle/working"))

In [ ]:
with open("adapter_config.json") as f:
    print(f.read())

In [ ]:
print(lora_path)

In [ ]:
# Generate predictions using continuous batching (with adapter)
if TEST_GENERATION:
    local_run_results, local_comparison_df = run_local_experiments(
        source_df=df,
        llm=llm,
        lora_path=lora_path,
        prompts=prompts,
        experiment_configs=LOCAL_EXPERIMENTS,
        primary_run_name=PRIMARY_RUN_NAME,
        error_rows=BASELINE_ERROR_ROWS,
    )
    df = local_run_results[PRIMARY_RUN_NAME]["eval_df"].copy()
    baseline_summary = local_run_results[PRIMARY_RUN_NAME]["summary"]
    print(local_comparison_df)


# Produce submission

In [ ]:
import zipfile as _zf

NON_SUBMISSION_FILES = {
    "submission.zip",
    "predictions.csv",
    "baseline_summary.json",
    "baseline_errors.csv",
}

print(os.listdir("."))
shutil.rmtree("reference", ignore_errors=True)
with _zf.ZipFile("submission.zip", "w", _zf.ZIP_DEFLATED) as zf:
    for file in os.listdir("."):
        if file.startswith(".") or file in NON_SUBMISSION_FILES or not os.path.isfile(file):
            continue
        zf.write(file)
        os.remove(file)


In [ ]:
if TEST_GENERATION:
    print(
        {
            "primary_run_name": PRIMARY_RUN_NAME,
            "comparison_path": "local_eval/experiment_compare.csv",
            "primary_predictions_path": "predictions.csv",
            "primary_summary_path": "baseline_summary.json",
            "primary_errors_path": "baseline_errors.csv",
        }
    )
    baseline_summary


In [ ]:
print(os.listdir("."))